# Validação A/B — portão de improdutividade

Faz dois replays: (1) o benchmark completo de 111 unidades compara a regra H6 do código remoto com o portão candidato; (2) o recorte de 79 eventos reaplica diretamente `backend/productivity.py` aos sinais estruturados exportados. O replay é conservador e não extrai decisão da descrição em texto. Abstenção é uma terceira saída e nunca é contada como acerto.

In [ ]:
import json, os, sys
from pathlib import Path
import pandas as pd

DATA_DIR = Path(os.environ.get('KV_VALIDACAO_DIR', '/content/drive/MyDrive/Spectra/produtividade'))
REPO_DIR = Path(os.environ.get('KV_REPO_DIR', Path.cwd())).resolve()
OUTPUT_DIR = Path(os.environ.get('KV_VALIDACAO_OUTPUT', DATA_DIR / 'resultado_portao_negativo')).resolve()
INPUT = DATA_DIR / 'alvo_ab_vlm.csv'
assert INPUT.exists(), f'Arquivo não encontrado: {INPUT}'
assert (REPO_DIR / 'backend' / 'productivity.py').exists(), f'Repositório não encontrado: {REPO_DIR}'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Dados:', INPUT)
print('Código candidato:', REPO_DIR)
print('Saída:', OUTPUT_DIR)

In [ ]:
sys.path.insert(0, str(REPO_DIR))
from backend.productivity import (
    EST_IMPRODUTIVO, EST_PRODUTIVO, classificar_observacao,
)

df = pd.read_csv(INPUT)
df['y_true'] = df['rotulo_humano'].astype(str).str.strip().str.upper()
df = df[df['y_true'].isin(['PRODUTIVO', 'IMPRODUTIVO'])].copy()
df['pred_A'] = df['sistema'].astype(str).str.strip().str.upper()
df['peso_s'] = pd.to_numeric(df.get('dur_s', 1.0), errors='coerce').fillna(1.0).clip(lower=0.0)

def bool_estrito(valor):
    if isinstance(valor, bool):
        return valor
    texto = str(valor).strip().lower()
    return True if texto == 'true' else False if texto == 'false' else None

def texto_ou_none(valor):
    if pd.isna(valor):
        return None
    texto = str(valor).strip()
    return texto or None

def predizer_candidato(linha):
    evento = {
        'papel_pessoa': 'operador',
        'maos_maquina': bool_estrito(linha.get('maos_maquina')),
        'orientacao': texto_ou_none(linha.get('orientacao')),
        'trabalho': bool_estrito(linha.get('trabalho')),
        'produtividade_motivo': texto_ou_none(linha.get('produtividade_motivo')),
        'comportamento_label': texto_ou_none(linha.get('label')),
        'descricao_bruta': texto_ou_none(linha.get('descricao')),
        '_cam_id': 'cam1',
    }
    estado, motivo = classificar_observacao(evento, frentes_por_camera={})
    pred = 'PRODUTIVO' if estado == EST_PRODUTIVO else 'IMPRODUTIVO' if estado == EST_IMPRODUTIVO else 'ABSTEM'
    return pd.Series({'pred_B': pred, 'motivo_B': motivo})

df[['pred_B', 'motivo_B']] = df.apply(predizer_candidato, axis=1)
print('Linhas avaliáveis:', len(df))
display(df[['sample_id', 'y_true', 'pred_A', 'pred_B', 'motivo_B']].head())

In [ ]:
def dividir(a, b):
    return None if b <= 0 else 100.0 * a / b

def metricas(dados, coluna):
    y, p, w = dados['y_true'], dados[coluna], dados['peso_s']
    soma = lambda mascara: float(w[mascara].sum())
    total = soma(pd.Series(True, index=dados.index))
    coberto = soma(p.isin(['PRODUTIVO', 'IMPRODUTIVO']))
    tp_i = soma((y == 'IMPRODUTIVO') & (p == 'IMPRODUTIVO'))
    fp_i = soma((y == 'PRODUTIVO') & (p == 'IMPRODUTIVO'))
    fn_i = soma((y == 'IMPRODUTIVO') & (p != 'IMPRODUTIVO'))
    tp_p = soma((y == 'PRODUTIVO') & (p == 'PRODUTIVO'))
    fp_p = soma((y == 'IMPRODUTIVO') & (p == 'PRODUTIVO'))
    fn_p = soma((y == 'PRODUTIVO') & (p != 'PRODUTIVO'))
    prec_i, rec_i = dividir(tp_i, tp_i + fp_i), dividir(tp_i, tp_i + fn_i)
    prec_p, rec_p = dividir(tp_p, tp_p + fp_p), dividir(tp_p, tp_p + fn_p)
    f1 = lambda pr, re: None if pr is None or re is None or pr + re == 0 else 2 * pr * re / (pr + re)
    corretos = soma(y == p)
    return {
        'precisao_improdutivo_pct': prec_i, 'recall_improdutivo_pct': rec_i,
        'f1_improdutivo_pct': f1(prec_i, rec_i),
        'precisao_produtivo_pct': prec_p, 'recall_produtivo_pct': rec_p,
        'f1_produtivo_pct': f1(prec_p, rec_p),
        'cobertura_pct': dividir(coberto, total),
        'abstencao_pct': dividir(total - coberto, total),
        'acuracia_operacional_pct': dividir(corretos, total),
        'acuracia_seletiva_pct': dividir(corretos, coberto),
        'taxa_falsa_acusacao_pct': dividir(fp_i, soma(y == 'PRODUTIVO')),
        'alegacoes_improdutivas_min': (tp_i + fp_i) / 60.0,
        'verdade_improdutiva_min': soma(y == 'IMPRODUTIVO') / 60.0,
        'total_min': total / 60.0,
    }

resultado = pd.DataFrame({
    'A_historico': metricas(df, 'pred_A'),
    'B_portao_negativo': metricas(df, 'pred_B'),
}).T
display(resultado.round(2))
print('Matriz A (contagem):')
display(pd.crosstab(df['y_true'], df['pred_A'], margins=True))
print('Matriz B (contagem):')
display(pd.crosstab(df['y_true'], df['pred_B'], margins=True))

In [ ]:
FULL_INPUT = DATA_DIR / 'unidades_avaliadas.csv'
H6_INPUT = DATA_DIR / 'predictions_B.csv'
resultado_completo = None
if FULL_INPUT.exists() and H6_INPUT.exists():
    completo = pd.read_csv(FULL_INPUT)
    h6 = pd.read_csv(H6_INPUT)[['unit_id', 'nivel_h6', 'motivo_h6']]
    completo = completo.merge(h6, on='unit_id', how='left')
    completo['peso_s'] = pd.to_numeric(completo['peso_s'], errors='coerce').fillna(0.0)
    completo['pred_C'] = completo['pred_B']
    origem_sem_atividade = completo['nivel_h6'].fillna('').isin(['identidade', 'presenca'])
    bloqueado = (completo['pred_C'] == 'IMPRODUTIVO') & origem_sem_atividade
    completo.loc[bloqueado, 'pred_C'] = 'ABSTEM'
    resultado_completo = pd.DataFrame({
        'B_H6_codigo_remoto': metricas(completo, 'pred_B'),
        'C_portao_origem': metricas(completo, 'pred_C'),
    }).T
    print('Benchmark completo — 111 unidades rotuladas:')
    display(resultado_completo.round(2))
    print('Alegações negativas bloqueadas por terem origem em identidade/presença:', int(bloqueado.sum()))
    display(pd.crosstab(completo['y_true'], completo['pred_C'], margins=True))
else:
    completo = None
    print('Benchmark completo não encontrado; mantendo apenas o replay direto de eventos.')

In [ ]:
motivo_disponivel = 'produtividade_motivo' in df.columns and df['produtividade_motivo'].notna().any()
alegacoes_b = int((df['pred_B'] == 'IMPRODUTIVO').sum())
print('O export contém produtividade_motivo?', motivo_disponivel)
print('Alegações improdutivas do candidato:', alegacoes_b)
if not motivo_disponivel:
    print('LIMITAÇÃO: o CSV histórico não preservou o motivo V9. O replay pode medir o ganho de segurança e a perda de cobertura, mas não estimar a precisão das novas alegações negativas.')
if alegacoes_b == 0:
    print('CONCLUSÃO: precisão de improdutividade do candidato é INDEFINIDA, não 100%. É obrigatório coletar motivos V12/V13 e rotular um conjunto cego antes de promover a regra por precisão.')

casos = [
    ('sem motivo', {'trabalho': False}),
    ('sem atividade explícita', {'trabalho': False, 'produtividade_motivo': 'sem_atividade'}),
    ('celular visível', {'trabalho': False, 'produtividade_motivo': 'uso_celular'}),
    ('motivo misto legado', {'trabalho': False, 'produtividade_motivo': 'conversa_ou_celular'}),
]
linhas = []
for nome, extra in casos:
    estado, motivo = classificar_observacao({'papel_pessoa': 'operador', **extra}, {})
    linhas.append({'caso': nome, 'estado': estado, 'motivo': motivo})
display(pd.DataFrame(linhas))

In [ ]:
df.to_csv(OUTPUT_DIR / 'predicoes_ab_portao_negativo.csv', index=False)
resultado.to_csv(OUTPUT_DIR / 'metricas_ab_portao_negativo.csv')
relatorio = {
    'linhas': int(len(df)),
    'motivo_historico_disponivel': bool(motivo_disponivel),
    'alegacoes_improdutivas_candidato': alegacoes_b,
    'metricas_replay_direto': json.loads(resultado.to_json(orient='index')),
    'metricas_benchmark_completo': (
        json.loads(resultado_completo.to_json(orient='index'))
        if resultado_completo is not None else None
    ),
    'interpretacao': (
        'Precisão negativa indefinida: o candidato não emitiu alegações improdutivas no export legado sem produtividade_motivo.'
        if alegacoes_b == 0 else
        'Comparação válida dentro do recorte exportado; confirmar em dias e turnos independentes antes de promoção.'
    ),
}
if resultado_completo is not None:
    completo.to_csv(OUTPUT_DIR / 'predicoes_benchmark_completo.csv', index=False)
    resultado_completo.to_csv(OUTPUT_DIR / 'metricas_benchmark_completo.csv')
(OUTPUT_DIR / 'relatorio_ab_portao_negativo.json').write_text(json.dumps(relatorio, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps(relatorio, ensure_ascii=False, indent=2))